## corner study
for the sake of time, we will not be looking at all corners over all the laps, instead we will try to take some corners that cover different types of corners for this track:
* C1 bc it's a wide fast
* [C2, C3, C4, C5] together bc they are esses
* C20 bc it's 90 degrees
* C24 and C27 bc hairpins
* C25 and C26 bc they're a snail corner

We will take the esses and the snail as one group and one window instead of two separate corners because it makes more sense.

and in the comparison, we will not take all these corners from all the laps, instead we will take:
* the corner of the reference lap (fastest lap)
* the fastest lap on that corner/group if it beats the reference one
* the slowest in stint 1
* the slowest in stint 2


In [1]:
from src.segment import load_segmented
from src.loader import load_centerline
from src.config import CORNER_GROUPS
from src.analysis import comparable_laps, reference_lap, corner_entries, group_times, pick_laps

In [2]:
data = load_segmented()
grid, laps, zones, apexes, gaps = data["grid"], data["laps"], data["zones"], data["apexes"], data["gaps"]
cl = load_centerline(); length = cl.attrs["length"]
laps_used = comparable_laps(laps)["lap"].tolist()
ref = reference_lap(laps)

entries = corner_entries(grid, apexes, zones, laps_used, length)
gt = group_times(grid, apexes, entries, zones, gaps, laps_used, CORNER_GROUPS, length)
picks = pick_laps(gt, ref)

In [3]:
picks

,group,role,lap,time,delta
0,C1,ref,10,4.083430,0.000000
1,C1,fastest,10,4.083430,0.000000
2,C1,slowest stint 1,3,4.930340,0.846910
3,C1,slowest stint 2,15,5.651795,1.568365
4,esses,ref,10,7.895925,0.000000
5,esses,fastest,22,7.595099,-0.300826
6,esses,slowest stint 1,3,8.898847,1.002922
7,esses,slowest stint 2,15,8.860281,0.964357
8,C20,ref,10,3.241829,0.000000
9,C20,fastest,21,3.063740,-0.178089


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.colors import qualitative
from plotly.subplots import make_subplots
from src.segment import U_COL, THROTTLE_COL, BRAKE_COL, STEER_COL

CHANNELS = [(U_COL, "speed (m/s)"), (THROTTLE_COL, "throttle (0–1)"),
            (BRAKE_COL, "brake front (bar)"), (STEER_COL, "steering (deg, + = left)")]
COAST = "coasting" # phase label for both pedals off
main_apex = apexes[apexes["is_main"]].set_index("corner")["s"]


def plot_group(name: str) -> go.Figure:
    p = picks[picks["group"] == name]
    roles = p.groupby("lap", sort=False)["role"].agg(" + ".join)    # ref + fastest merged if same lap
    w = gt[gt["group"] == name].iloc[0]
    s0, s1 = float(w["s_start"]), float(w["s_end"])
    span = np.mod(s1 - s0, length)
    corners = CORNER_GROUPS[name]

    fig = make_subplots(rows=len(CHANNELS), cols=1, shared_xaxes=True, vertical_spacing=0.03)
    for k, (lap, role) in enumerate(roles.items()):
        lg = grid[grid["lap"] == lap].copy()
        lg["d"] = np.mod(lg["s"] - s0, length)                       # metres from window start
        lg = lg[lg["d"] <= span].sort_values("d").reset_index(drop=True)
        color = qualitative.Plotly[k % len(qualitative.Plotly)]
        label = f"lap {lap} ({role})"

        coast = (lg["phase"] == COAST).to_numpy()
        coast_ext = coast | np.roll(coast, 1) | np.roll(coast, -1)   # overlap one point so lines join
        lap_entries = entries[(entries["lap"] == lap) & (entries["corner"].isin(corners))]

        for i, (col, ylabel) in enumerate(CHANNELS, start=1):
            y = lg[col].to_numpy()
            fig.add_trace(go.Scatter(x=lg["d"], y=np.where(coast, np.nan, y), mode="lines",
                                     line=dict(color=color), name=label, legendgroup=str(lap),
                                     showlegend=(i == 1)), row=i, col=1)
            fig.add_trace(go.Scatter(x=lg["d"], y=np.where(coast_ext, y, np.nan), mode="lines",
                                     line=dict(color=color, dash="dash"), name=f"{label} coasting",
                                     legendgroup=str(lap), showlegend=False), row=i, col=1)

            for _, e in lap_entries.iterrows():                      # braking point / corner entry
                xe = float(np.mod(e["s_entry"] - s0, length))
                fig.add_trace(go.Scatter(x=[xe], y=[np.interp(xe, lg["d"], y)], mode="markers",
                                         marker=dict(color=color, size=11, symbol="triangle-down",
                                                     line=dict(color="black", width=1)),
                                         name=label, legendgroup=str(lap), showlegend=False,
                                         hovertext=f"lap {lap}: C{int(e['corner'])} entry ({e['entry_source']})",
                                         hoverinfo="text"), row=i, col=1)
            fig.update_yaxes(title_text=ylabel, row=i, col=1)

        u_exit = float(lg[U_COL].iloc[-1])                           # exit speed at window end
        fig.add_trace(go.Scatter(x=[float(lg["d"].iloc[-1])], y=[u_exit], mode="markers",
                                 marker=dict(color=color, size=12, symbol="diamond",
                                             line=dict(color="black", width=1)),
                                 name=label, legendgroup=str(lap), showlegend=False,
                                 hovertext=f"lap {lap}: exit {u_exit:.2f} m/s", hoverinfo="text"),
                      row=1, col=1)

    for c in corners:                                                # main apexes of the group
        x = float(np.mod(main_apex[c] - s0, length))
        fig.add_vline(x=x, line_dash="dot", line_color="grey")
        fig.add_annotation(x=x, y=1.0, yref="paper", text=f"C{c} apex", showarrow=False, yanchor="bottom")

    # legend keys for the symbols
    for sym, txt in [("triangle-down", "entry / braking point"), ("diamond", "exit speed")]:
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=txt,
                                 marker=dict(color="grey", size=11, symbol=sym)), row=1, col=1)
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", name="coasting",
                             line=dict(color="grey", dash="dash")), row=1, col=1)

    fig.update_xaxes(title_text="metres from window start", row=len(CHANNELS), col=1)
    fig.update_layout(title=f"{name}: picked laps", height=1050, width=1300,
                      legend=dict(orientation="h", y=1.07))
    return fig


In [ ]:
# group C1, wide 
plot_group("C1").show()

Comparison of laps 3 and 15 with lap 10:
* Lap 15: there doesn't seem to be any braking point; the driver instead opted to lift at around 16.5 m. The driver was already entering the corner slowly, at 8.7 m/s and rising, and lifted when he reached around 11 m/s. After the apex the driver presses back on the throttle very little, and the speed picks up very gradually until he exits the corner at 13.85 m/s.
* Lap 3 on the other hand, brakes, and does it much earlier, at around 2.7 m from the start of the window. We notice that the driver coasts before hitting the brakes, so most likely they started lifting, coasted for 1.5 m and then hit the brakes, all this at 12.7 m/s. The transition between lifting and braking was not smooth: we expect the driver to lift off the throttle and press the brakes gradually, whereas here it looks more like they lifted off completely and then started braking. The braking pattern does not seem smooth at all either. We usually expect the trace to go up and then back down once, but instead it went up, then down, then up again and then down, so it seems like the driver was hesitant. The driver then lifts off the brakes at 12.7 m from the window start, at 11 m/s. The coasting lasts about 10 m, then the driver touches the throttle a tiny bit right before the apex and starts actually accelerating, with an upward trend, at 26.1 m (after the apex). The max throttle reaches 0.61, at a speed of 9.8 m/s, which then increases gradually, and he exits at 19.33 m/s.
* Lap 10: the driver enters at 15.2 m/s with the throttle at 0.73. He then starts lifting off the throttle and braking at the same time, gradually and smoothly, and starts to turn the car as well. The max braking reaches 9.5 bar and the throttle reaches 0. The braking lasts 13 m, with 1 m of coasting between ~16.4 and ~17.4 m. The minimum speed is 12.2 m/s and is reached before the apex, and the speed is maintained through the corner's apex. The driver starts pressing the throttle gradually before the apex, at 18.6 m, and it goes up to 0.6 at 30 m. Then the driver lifts a little and goes back on the throttle at 37.8 m until it reaches 0.78, and it stays fairly steady with the speed rising until he exits at 21.59 m/s. The hesitation here on the throttle might have cost him some time, though.

* Additional notes:
  - A lot of the loss comes from C27, before C1 starts.